# JobShield
## 02 — Data Preprocessing

### Objectives
1. Load and validate the EMSCAD dataset.
2. Handle missing information.
3. Identify and remove duplicate postings.
4. Exclude the job identifier.
5. Create stratified training, validation and test sets.
6. Save the prepared datasets.

### Data Leakage Prevention
Learned preprocessing transformations must be fitted only
on training data.

Missing values will be replaced using fixed rules that
do not learn statistical information from the dataset.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from IPython.display import display


def find_dataset():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        path = folder / "data" / "fake_job_postings.csv"

        if path.is_file():
            return path

    raise FileNotFoundError(
        "Cannot find data/fake_job_postings.csv"
    )


dataset_path = find_dataset()
df = pd.read_csv(dataset_path)

TEXT_COLS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

CAT_COLS = [
    "location",
    "department",
    "salary_range",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

BINARY_COLS = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

TARGET = "fraudulent"
FEATURE_COLS = TEXT_COLS + CAT_COLS + BINARY_COLS

expected_columns = set(
    FEATURE_COLS + ["job_id", TARGET]
)

if set(df.columns) != expected_columns:
    raise ValueError("Unexpected dataset columns.")

print("Dataset loaded successfully.")
print("Original shape:", df.shape)

display(df.head(3))

Dataset loaded successfully.
Original shape: (17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


## 1. Missing-Value Handling

The dataset contains several fields with missing information.

For this phase:

- Missing text will become an empty string.
- Missing categorical information will become "Missing".
- Binary indicators and target labels will be validated.

We will retain incomplete postings because missing information
may itself be useful when detecting fraudulent advertisements.

These fixed replacement rules do not calculate any statistics
from the training, validation or test sets.

In [4]:
def clean_missing_values(data):

    result = data.copy()

    # Handle textual information
    for column in TEXT_COLS:
        result[column] = (
            result[column]
            .fillna("")
            .astype(str)
        )

    # Handle categorical information
    for column in CAT_COLS:
        result[column] = (
            result[column]
            .fillna("Missing")
            .astype(str)
            .str.strip()
            .replace("", "Missing")
        )

    # Validate binary indicators
    for column in BINARY_COLS:
        result[column] = pd.to_numeric(
            result[column],
            errors="raise"
        ).astype(int)

        if not result[column].isin([0, 1]).all():
            raise ValueError(
                f"Invalid binary values in {column}"
            )

    # Validate target labels
    result[TARGET] = pd.to_numeric(
        result[TARGET],
        errors="raise"
    ).astype(int)

    if not result[TARGET].isin([0, 1]).all():
        raise ValueError("Invalid target labels.")

    return result


cleaned_df = clean_missing_values(df)

print("Missing values remaining:")
print(cleaned_df.isna().sum().sum())

display(cleaned_df.head())

Missing values remaining:
0


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,Missing,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,1,0,Other,Internship,Missing,Missing,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,Missing,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,Missing,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",Missing,Missing,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,0,1,0,Missing,Missing,Missing,Missing,Missing,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,Missing,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",Missing,Missing,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


## 2. Duplicate Record Removal

The job_id column is a record identifier rather than a
meaningful predictor of recruitment fraud.

We will remove repeated advertisements by comparing their
features without the job identifier.

Before removing duplicates, we will check whether identical
feature records have conflicting target labels.

This step takes place before splitting the dataset to reduce
the risk of identical postings appearing across data splits.

In [5]:
original_count = len(cleaned_df)

# Check whether identical features have conflicting labels
label_counts = (
    cleaned_df
    .groupby(FEATURE_COLS, dropna=False)[TARGET]
    .nunique()
)

conflicting_groups = int(
    (label_counts > 1).sum()
)

print("Conflicting label groups:", conflicting_groups)

if conflicting_groups > 0:
    raise ValueError(
        "Identical feature records have conflicting labels."
    )

# Remove repeated postings
cleaned_df = cleaned_df.drop_duplicates(
    subset=FEATURE_COLS,
    keep="first"
).copy()

removed_count = original_count - len(cleaned_df)

# Exclude the unique identifier
cleaned_df = cleaned_df.drop(
    columns=["job_id"]
).reset_index(drop=True)

print("Original records:", original_count)
print("Removed duplicates:", removed_count)
print("Remaining records:", len(cleaned_df))

print("\nClass distribution:")
display(
    cleaned_df[TARGET]
    .value_counts()
    .rename(index={0: "Legitimate", 1: "Fraudulent"})
)

Conflicting label groups: 0
Original records: 17880
Removed duplicates: 281
Remaining records: 17599

Class distribution:


fraudulent
Legitimate    16743
Fraudulent      856
Name: count, dtype: int64

## 3. Stratified Train, Validation and Test Split

The prepared data will be divided into:

- Training: 70%
- Validation: 15%
- Testing: 15%

Stratification preserves approximately the same class
distribution across the three datasets.

A fixed random state makes the split reproducible.

The test dataset will remain untouched during model
development, feature selection and threshold tuning.

In [6]:
RANDOM_STATE = 42

# First reserve 15% for testing
train_val_df, test_df = train_test_split(
    cleaned_df,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=cleaned_df[TARGET]
)

# Divide the remaining 85% into 70% and 15%
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE,
    stratify=train_val_df[TARGET]
)

# Verify there is no overlap
assert set(train_df.index).isdisjoint(val_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(val_df.index).isdisjoint(test_df.index)

assert (
    len(train_df) + len(val_df) + len(test_df)
    == len(cleaned_df)
)

def split_summary(name, data):
    return {
        "Dataset": name,
        "Total": len(data),
        "Legitimate": int((data[TARGET] == 0).sum()),
        "Fraudulent": int((data[TARGET] == 1).sum()),
        "Fraud %": round(data[TARGET].mean() * 100, 2)
    }


summary = pd.DataFrame([
    split_summary("Training", train_df),
    split_summary("Validation", val_df),
    split_summary("Testing", test_df)
])

display(summary)

,Dataset,Total,Legitimate,Fraudulent,Fraud %
0,Training,12319,11719,600,4.87
1,Validation,2640,2512,128,4.85
2,Testing,2640,2512,128,4.85


## 4. Saving Prepared Data

The cleaned and split datasets will be saved into
data/processed/.

These files will be used for feature engineering,
DNN training and final evaluation.

No learned feature transformation has been fitted
at this stage.

In [7]:
processed_dir = dataset_path.parent / "processed"

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

train_path = processed_dir / "train.csv"
val_path = processed_dir / "validation.csv"
test_path = processed_dir / "test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Prepared datasets saved successfully.")

for path in [train_path, val_path, test_path]:
    print(path)

# Verify saved row counts
for path, expected_length in [
    (train_path, len(train_df)),
    (val_path, len(val_df)),
    (test_path, len(test_df))
]:
    saved = pd.read_csv(path)

    assert len(saved) == expected_length

    assert "job_id" not in saved.columns

    assert TARGET in saved.columns

print("\nPASS: Saved file validation completed.")

Prepared datasets saved successfully.
d:\the_land_of_the_code\college index\ANN-LAB\FINAL PROJECT\data\processed\train.csv
d:\the_land_of_the_code\college index\ANN-LAB\FINAL PROJECT\data\processed\validation.csv
d:\the_land_of_the_code\college index\ANN-LAB\FINAL PROJECT\data\processed\test.csv

PASS: Saved file validation completed.


## 5. Preprocessing Conclusion

The dataset has been prepared for feature engineering.

### Completed
- Validated dataset structure.
- Handled missing information.
- Removed duplicate postings.
- Removed the job identifier.
- Performed reproducible stratified splitting.
- Saved training, validation and test datasets.

### Important Note
The TF-IDF vectorizer, categorical encoder and any scaler
will be fitted using training data only.

Validation and test records will be transformed using
those same fitted training artefacts.

### Next Phase
03 — Feature Engineering and Baseline Model